SCENARIO: “Retail Smart Assistant with Role-Based Access”
🏬 Background Story
A large retail chain introduces an AI-powered store assistant to streamline operations.
👉 Users can ask:
- “What is my purchase history?”
- “Check inventory for product X.”
- “Approve supplier orders.”
- “Manage employee schedules.”
👉 But not everyone can do everything — access depends on roles

In [ ]:
!pip install groq gradio nest_asyncio

import os
import asyncio
import nest_asyncio
import gradio as gr
from groq import Groq

os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

nest_asyncio.apply()

_original_asyncio_run = asyncio.run

def compatible_asyncio_run(main, *, debug=None, loop_factory=None):
    return _original_asyncio_run(main)

asyncio.run = compatible_asyncio_run


users = {
    "C101": {
        "role": "customer",
        "name": "Rahul Sharma",
        "purchase_history": [
            {"date": "2026-03-20", "item": "Running Shoes", "amount": "₹2,499"},
            {"date": "2026-03-15", "item": "T-Shirt", "amount": "₹799"},
            {"date": "2026-03-10", "item": "Backpack", "amount": "₹1,299"}
        ]
    },
    "C102": {
        "role": "customer",
        "name": "Priya Verma",
        "purchase_history": [
            {"date": "2026-03-18", "item": "Skin Care Kit", "amount": "₹1,899"},
            {"date": "2026-03-14", "item": "Handbag", "amount": "₹2,999"},
            {"date": "2026-03-08", "item": "Perfume", "amount": "₹1,499"}
        ]
    },
    "S201": {
        "role": "staff",
        "name": "Amit Singh",
        "purchase_history": [
            {"date": "2026-03-11", "item": "Staff Uniform", "amount": "₹1,200"}
        ],
        "department": "Sales Floor"
    },
    "S202": {
        "role": "staff",
        "name": "Neha Jain",
        "purchase_history": [
            {"date": "2026-03-09", "item": "Store Shoes", "amount": "₹1,050"}
        ],
        "department": "Inventory"
    },
    "M301": {
        "role": "manager",
        "name": "Store Manager",
        "purchase_history": [
            {"date": "2026-03-05", "item": "Office Supplies", "amount": "₹2,250"}
        ],
        "store": "Retail Hub - Noida"
    }
}


purchase_master_records = {
    "Rahul Sharma": [
        {"date": "2026-03-20", "item": "Running Shoes", "amount": "₹2,499"},
        {"date": "2026-03-15", "item": "T-Shirt", "amount": "₹799"},
        {"date": "2026-03-10", "item": "Backpack", "amount": "₹1,299"}
    ],
    "Priya Verma": [
        {"date": "2026-03-18", "item": "Skin Care Kit", "amount": "₹1,899"},
        {"date": "2026-03-14", "item": "Handbag", "amount": "₹2,999"},
        {"date": "2026-03-08", "item": "Perfume", "amount": "₹1,499"}
    ],
    "Rohan Mehta": [
        {"date": "2026-03-16", "item": "Laptop Bag", "amount": "₹1,799"},
        {"date": "2026-03-12", "item": "Notebook Set", "amount": "₹499"}
    ]
}


inventory_data = {
    "Laptop": {"stock": 24, "status": "In Stock"},
    "Running Shoes": {"stock": 12, "status": "In Stock"},
    "Perfume": {"stock": 6, "status": "Low Stock"},
    "Handbag": {"stock": 3, "status": "Low Stock"},
    "T-Shirt": {"stock": 31, "status": "In Stock"}
}


supplier_orders = [
    {"order_id": "SO101", "supplier": "ABC Distributors", "item": "Laptop", "status": "Pending Approval"},
    {"order_id": "SO102", "supplier": "Fashion World", "item": "Handbag", "status": "Pending Approval"}
]


employee_schedules = [
    {"employee": "Amit Singh", "shift": "9 AM - 5 PM", "day": "Monday"},
    {"employee": "Neha Jain", "shift": "10 AM - 6 PM", "day": "Monday"},
    {"employee": "Ritu Kapoor", "shift": "1 PM - 9 PM", "day": "Tuesday"}
]


permissions = {
    "customer": [
        "view_own_purchase_history"
    ],
    "staff": [
        "view_own_purchase_history",
        "view_others_purchases",
        "check_inventory_levels"
    ],
    "manager": [
        "view_own_purchase_history",
        "view_others_purchases",
        "approve_supplier_orders",
        "check_inventory_levels",
        "manage_employee_schedules"
    ]
}


intent_permission_map = {
    "own_purchase_history": "view_own_purchase_history",
    "others_purchases": "view_others_purchases",
    "approve_supplier_orders": "approve_supplier_orders",
    "check_inventory": "check_inventory_levels",
    "manage_schedules": "manage_employee_schedules",
    "full_summary": None
}


async def get_own_purchase_history(user_id):
    await asyncio.sleep(1)
    user = users[user_id]
    text = (
        f"Own Purchase History:\n"
        f"- Name: {user['name']}\n"
        f"- Role: {user['role'].title()}\n"
    )
    for purchase in user["purchase_history"]:
        text += f"- Date: {purchase['date']}, Item: {purchase['item']}, Amount: {purchase['amount']}\n"
    return text.strip()


async def get_others_purchases(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role not in ["staff", "manager"]:
        return "Access denied."

    text = "Others' Purchase Records:\n"
    for customer_name, purchases in purchase_master_records.items():
        text += f"- Customer: {customer_name}\n"
        for purchase in purchases:
            text += f"  Date: {purchase['date']}, Item: {purchase['item']}, Amount: {purchase['amount']}\n"
    return text.strip()


async def approve_supplier_orders_data(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role != "manager":
        return "Access denied."

    text = f"Supplier Order Approval Access:\n- Manager: {users[user_id]['name']}\n"
    for order in supplier_orders:
        text += (
            f"- Order ID: {order['order_id']}, Supplier: {order['supplier']}, "
            f"Item: {order['item']}, Status: {order['status']}\n"
        )
    return text.strip()


async def check_inventory_data(product_name=None):
    await asyncio.sleep(1)

    if product_name:
        product_name = product_name.strip()
        if product_name in inventory_data:
            item = inventory_data[product_name]
            return (
                f"Inventory Details:\n"
                f"- Product: {product_name}\n"
                f"- Stock: {item['stock']}\n"
                f"- Status: {item['status']}"
            )
        return f"Product '{product_name}' not found in inventory."

    text = "Inventory Levels:\n"
    for product, details in inventory_data.items():
        text += f"- Product: {product}, Stock: {details['stock']}, Status: {details['status']}\n"
    return text.strip()


async def manage_employee_schedules_data(user_id):
    await asyncio.sleep(1)
    role = users[user_id]["role"]

    if role != "manager":
        return "Access denied."

    text = f"Employee Schedule Management:\n- Manager: {users[user_id]['name']}\n"
    for schedule in employee_schedules:
        text += (
            f"- Employee: {schedule['employee']}, "
            f"Shift: {schedule['shift']}, Day: {schedule['day']}\n"
        )
    return text.strip()


async def parallel_role_data_fetch(user_id, product_name=None):
    role = users[user_id]["role"]

    if role == "customer":
        results = await asyncio.gather(
            get_own_purchase_history(user_id),
            return_exceptions=True
        )
        return {
            "own_purchase_history": results[0] if not isinstance(results[0], Exception) else "Purchase history unavailable"
        }

    if role == "staff":
        results = await asyncio.gather(
            get_own_purchase_history(user_id),
            get_others_purchases(user_id),
            check_inventory_data(product_name),
            return_exceptions=True
        )
        return {
            "own_purchase_history": results[0] if not isinstance(results[0], Exception) else "Purchase history unavailable",
            "others_purchases": results[1] if not isinstance(results[1], Exception) else "Other purchase data unavailable",
            "check_inventory": results[2] if not isinstance(results[2], Exception) else "Inventory data unavailable"
        }

    if role == "manager":
        results = await asyncio.gather(
            get_own_purchase_history(user_id),
            get_others_purchases(user_id),
            approve_supplier_orders_data(user_id),
            check_inventory_data(product_name),
            manage_employee_schedules_data(user_id),
            return_exceptions=True
        )
        return {
            "own_purchase_history": results[0] if not isinstance(results[0], Exception) else "Purchase history unavailable",
            "others_purchases": results[1] if not isinstance(results[1], Exception) else "Other purchase data unavailable",
            "approve_supplier_orders": results[2] if not isinstance(results[2], Exception) else "Supplier order data unavailable",
            "check_inventory": results[3] if not isinstance(results[3], Exception) else "Inventory data unavailable",
            "manage_schedules": results[4] if not isinstance(results[4], Exception) else "Schedule data unavailable"
        }

    return {}


def decide_intent(user_query, role):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
You are a retail smart assistant intent classifier.

Current role: {role}

Allowed intents for customer:
- own_purchase_history
- full_summary

Allowed intents for staff:
- own_purchase_history
- others_purchases
- check_inventory
- full_summary

Allowed intents for manager:
- own_purchase_history
- others_purchases
- approve_supplier_orders
- check_inventory
- manage_schedules
- full_summary

Rules:
- "What is my purchase history?" -> own_purchase_history
- "Show others purchases" -> others_purchases
- "Check inventory for product X" -> check_inventory
- "Approve supplier orders" -> approve_supplier_orders
- "Manage employee schedules" -> manage_schedules
- "Give me full summary" -> full_summary

Return exactly one valid intent for the current role.
If the role is not allowed, still choose the closest intent so the system can deny access properly.

User Query: {user_query}
"""
        }]
    )
    return response.choices[0].message.content.strip().lower()


def extract_product_name(user_query):
    products = list(inventory_data.keys())
    for product in products:
        if product.lower() in user_query.lower():
            return product
    return None


def analyse_retail_data(text, role):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Analyze this retail RBAC-based data for a {role} user and provide:
1. Key retail summary
2. Important observations
3. Role-based access note
4. Operational insight
5. Simple user-friendly explanation

Data:
{text}
"""
        }]
    )
    return response.choices[0].message.content


def generate_retail_report(analysis, user_id):
    user_name = users.get(user_id, {}).get("name", "User")
    user_role = users.get(user_id, {}).get("role", "unknown")

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Create a professional but simple retail smart assistant report for {user_name}.

Role: {user_role}

Use this analysis:
{analysis}

Keep it:
- clear
- structured
- role-aware
- concise but informative
"""
        }]
    )
    return response.choices[0].message.content


async def full_pipeline(user_id, user_query):
    if user_id not in users:
        return "Invalid User ID. Please enter C101, C102, S201, S202, or M301."

    role = users[user_id]["role"]
    intent = decide_intent(user_query, role)
    product_name = extract_product_name(user_query)
    data = await parallel_role_data_fetch(user_id, product_name)

    required_permission = intent_permission_map.get(intent)

    if required_permission is not None and required_permission not in permissions[role]:
        return (
            "==============================\n"
            "RETAIL SMART ASSISTANT OUTPUT\n"
            "==============================\n\n"
            f"User ID: {user_id}\n"
            f"Role: {role}\n"
            f"Detected Intent: {intent}\n\n"
            f"Access Denied: {role} cannot perform '{required_permission}'."
        )

    if role == "customer":
        if intent == "own_purchase_history":
            combined_text = data["own_purchase_history"]
        elif intent == "full_summary":
            combined_text = "\n\n".join(data.values())
        else:
            return (
                "==============================\n"
                "RETAIL SMART ASSISTANT OUTPUT\n"
                "==============================\n\n"
                f"User ID: {user_id}\n"
                f"Role: {role}\n"
                f"Detected Intent: {intent}\n\n"
                "Access Denied: Customer role has limited permissions based on RBAC policy."
            )

    elif role == "staff":
        if intent == "own_purchase_history":
            combined_text = data["own_purchase_history"]
        elif intent == "others_purchases":
            combined_text = data["others_purchases"]
        elif intent == "check_inventory":
            combined_text = data["check_inventory"]
        else:
            combined_text = "\n\n".join(data.values())

    elif role == "manager":
        if intent == "own_purchase_history":
            combined_text = data["own_purchase_history"]
        elif intent == "others_purchases":
            combined_text = data["others_purchases"]
        elif intent == "approve_supplier_orders":
            combined_text = data["approve_supplier_orders"]
        elif intent == "check_inventory":
            combined_text = data["check_inventory"]
        elif intent == "manage_schedules":
            combined_text = data["manage_schedules"]
        else:
            combined_text = "\n\n".join(data.values())

    else:
        return "Invalid role."

    analysis = analyse_retail_data(combined_text, role)
    report = generate_retail_report(analysis, user_id)

    final_output = f"""
==============================
RETAIL SMART ASSISTANT OUTPUT
==============================

User ID: {user_id}
Role: {role}
Detected Intent: {intent}

RAW DATA:
{combined_text}

--------------------------------
AI ANALYSIS + ROLE-BASED REPORT:
--------------------------------
{report}
"""
    return final_output.strip()


def run_normal_mode_logic(user_id, user_question):
    user_id = user_id.strip()
    user_question = user_question.strip()

    if not user_id or not user_question:
        return "Please enter both User ID and question."

    return asyncio.run(full_pipeline(user_id, user_question))


def retail_assistant_ui(user_id, user_query):
    user_id = user_id.strip()
    user_query = user_query.strip()

    if not user_id or not user_query:
        return "Please enter both User ID and question."

    return asyncio.run(full_pipeline(user_id, user_query))


with gr.Blocks() as demo:
    gr.Markdown("# Retail Smart Assistant with Role-Based Access")
    gr.Markdown("""
Use IDs by role:

Customer IDs:
- C101
- C102

Staff IDs:
- S201
- S202

Manager ID:
- M301

Example questions:
- What is my purchase history?
- Show others purchases.
- Check inventory for Laptop.
- Approve supplier orders.
- Manage employee schedules.
- Give me full summary.
""")

    with gr.Tab("Normal Input Mode"):
        normal_user_id = gr.Textbox(
            label="Enter User ID",
            placeholder="Example: C101 or S201 or M301"
        )
        normal_query = gr.Textbox(
            label="Ask your question",
            placeholder="Example: What is my purchase history?"
        )
        normal_output = gr.Textbox(
            label="Final Output",
            lines=24
        )
        normal_btn = gr.Button("Run Normal Mode")
        normal_btn.click(
            fn=run_normal_mode_logic,
            inputs=[normal_user_id, normal_query],
            outputs=normal_output
        )

    with gr.Tab("Gradio Real-Time Mode"):
        user_id_input = gr.Textbox(
            label="Enter User ID",
            placeholder="Example: C101 or S201 or M301"
        )
        query_input = gr.Textbox(
            label="Ask your question",
            placeholder="Example: Check inventory for Laptop"
        )
        output_box = gr.Textbox(
            label="Assistant Response",
            lines=24
        )
        submit_btn = gr.Button("Get Details")
        submit_btn.click(
            fn=retail_assistant_ui,
            inputs=[user_id_input, query_input],
            outputs=output_box
        )

demo.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.6 MB/s eta 0:00:00
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8f3a2391a0b6659661.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
